# Diabetes Prediction - Categorical Representation

**Portfolio version.** Notebook outputs were cleared to avoid publishing row-level data. The dataset is intentionally excluded from the repository; place `frankfurt_diabetes.csv` in `03_Data/` before running.


In [ ]:
import os
import pandas as pd
import numpy as np
import pandas as pd


df = pd.read_csv("../03_Data/frankfurt_diabetes.csv")

In [ ]:
# Cek duplikasi baris pada dataset mentah
print("Jumlah baris total :", len(df))
print("Jumlah baris unik   :", len(df.drop_duplicates()))
print("Jumlah baris duplikat:", df.duplicated().sum())

# Kalau ada duplikat, lihat contohnya
if df.duplicated().sum() > 0:
    print(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(20))

In [ ]:
# 1. Ambil copy dataframe Frankfurt
df_rekateg = df.copy()

# 2. Handling Missing Values (Nilai 0 pada variabel fisiologis diubah ke NaN sementara)
cols_missing = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]
df_temp = df_rekateg.copy()
df_temp[cols_missing] = df_temp[cols_missing].replace(0, np.nan)

In [ ]:
df_rekateg["Pregnancies_Cat"] = pd.cut(
    df_rekateg["Pregnancies"],
    bins=[-1, 0, 2, df_rekateg["Pregnancies"].max()],
    labels=["Nulipara (0)", "Primipara/Multipara Rendah (1-2)", "Grandemultipara (>=3)"],
)
df_rekateg["BloodPressure_Cat"] = pd.cut(
    df_temp["BloodPressure"],
    bins=[-1, 79, df_temp["BloodPressure"].max()],
    labels=["Normal (<80)", "Hipertensi (>=80)"],
)
print(df_rekateg['BloodPressure_Cat'].value_counts())
med_skin = df_temp["SkinThickness"].median()

df_rekateg["SkinThickness_Cat"] = pd.cut(
    df_temp["SkinThickness"],
    bins=[-1, med_skin - 0.01, df_temp["SkinThickness"].max()],
    labels=[f"Rendah (<{med_skin:.0f} mm)", f"Tinggi (>={med_skin:.0f} mm)"]
)
print(df_rekateg['SkinThickness_Cat'].value_counts())
df_rekateg["Insulin_Cat"] = pd.cut(
    df_rekateg["Insulin"],
    bins=[-1, 0, df_rekateg["Insulin"].max()+1],
    labels=["Tidak Ada Insulin (0)", "Ada Insulin (>0)"]
)
print(df_rekateg['Insulin_Cat'].value_counts())
df_rekateg["BMI_Cat"] = pd.cut(
    df_temp["BMI"],
    bins=[-1, 18.49, 24.99, 29.99, df_temp["BMI"].max()],
    labels=["Underweight (<18.5)", "Normal (18.5-24.9)", "Overweight (25-29.9)", "Obese (>=30)"]
)
print(df_rekateg['BMI_Cat'].value_counts())
med_dpf = df_rekateg["DiabetesPedigreeFunction"].median()

df_rekateg["DPF_Cat"] = pd.cut(
    df_rekateg["DiabetesPedigreeFunction"],
    bins=[-1, med_dpf, df_rekateg["DiabetesPedigreeFunction"].max()],
    labels=[f"Genetik Rendah (<{med_dpf:.3f})", f"Genetik Tinggi (>={med_dpf:.3f})"],
)
print(df_rekateg['DPF_Cat'].value_counts())
df_rekateg["Age_Cat"] = pd.cut(
    df_rekateg["Age"],
    bins=[-1, 34, df_rekateg["Age"].max()],
    labels=["Berisiko Rendah (<35)", "Berisiko Tinggi (>=35)"]
)
print(df_rekateg['Age_Cat'].value_counts())

In [ ]:
df_rekateg["BloodPressure_Cat"] = pd.cut(
    df_rekateg["BloodPressure"],
    bins=[-1, 0, 79, df_rekateg["BloodPressure"].max()],
    labels=["Tidak Terukur (0)", "Normal (<80)", "Hipertensi (>=80)"]
)

# 2. SkinThickness (0 dimasukkan ke kategori tersendiri)
df_rekateg["SkinThickness_Cat"] = pd.cut(
    df_rekateg["SkinThickness"],
    bins=[-1, 0, 24.9, df_rekateg["SkinThickness"].max()],
    labels=["Tidak Terukur (0)", "Normal (<25 mm)", "Tinggi (>=25 mm)"]
)

# 3. BMI (0 dimasukkan ke kategori tersendiri)
df_rekateg["BMI_Cat"] = pd.cut(
    df_rekateg["BMI"],
    bins=[-1, 0, 18.49, 24.99, 29.99, df_rekateg["BMI"].max()],
    labels=["Tidak Terukur (0)", "Underweight", "Normal", "Overweight", "Obese"]
)

In [ ]:
df_rekateg.info()

In [ ]:
df_rekateg.columns = df_rekateg.columns.str.lower()

In [ ]:
import pandas as pot

# 2. Ambil hanya kolom kategorik (_cat) dan target (outcome)
cols_to_keep = [col for col in df_rekateg.columns if col.endswith('_cat') or col == 'outcome']
df_clean_cat = df_rekateg[cols_to_keep].copy()

# 3. Cek struktur dataset baru
print("Daftar Kolom Dataset Baru:")
print(df_clean_cat.columns.tolist())

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import sklearn.metrics as metrics
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# ==========================================
# 1. LOAD DATA & CONVERT TO CATEGORY
# ==========================================
df = df_clean_cat.copy()

# 1. Pilih semua kolom kategorik (yang berakhiran _cat)
cat_cols = [col for col in df.columns if col.endswith("_cat")]

# 2. Ubah tipe datanya menjadi 'category'
df[cat_cols] = df[cat_cols].astype("category")

# Separasi Target (y) dan Fitur (X)
TARGET_COL = "outcome"
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Split Train-Test Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=6
)

# Background Data untuk SHAP Explainer
Xbackground = shap.utils.sample(X_train, nsamples=500, random_state=3454)


# ==========================================
# 2. XGBOOST CLASSIFIER (Native Categorical)
# ==========================================
print("\n=== XGBoost Model ===")
# Parameter enable_categorical=True wajib digunakan jika X berupa dtype 'category'
xgb_mod = xgb.XGBClassifier(enable_categorical=True, random_state=342)
xgb_mod.fit(X_train, y_train)

y_pred_xgb = xgb_mod.predict(X_test)
print(metrics.classification_report(y_test, y_pred_xgb))

# SHAP Explainer XGBoost
explainer_xgb = shap.TreeExplainer(xgb_mod)
xgb_shap_vals = explainer_xgb(X_test)

# Visualisasi SHAP
shap.plots.bar(xgb_shap_vals)
shap.plots.beeswarm(xgb_shap_vals)
shap.plots.waterfall(xgb_shap_vals[0])


# ==========================================
# 3. PREPARATION FOR SKLEARN MODELS (RF & MLP)
# ==========================================
# Karena Scikit-Learn belum mendukung dtype 'category', kita konversi ke kode angka (integer)
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

for col in cat_cols:
  X_train_encoded[col] = X_train_encoded[col].cat.codes
  X_test_encoded[col] = X_test_encoded[col].cat.codes

Xbackground_encoded = shap.utils.sample(
    X_train_encoded, nsamples=500, random_state=3454
)


# ==========================================
# 4. RANDOM FOREST CLASSIFIER
# ==========================================
print("\n=== Random Forest Model ===")
rf_mod = RandomForestClassifier(n_estimators=100, random_state=42342)
rf_mod.fit(X_train_encoded, y_train)

y_pred_rf = rf_mod.predict(X_test_encoded)
print(metrics.classification_report(y_test, y_pred_rf))

explainer_rf = shap.TreeExplainer(rf_mod, Xbackground_encoded)
rf_shap_vals = explainer_rf(X_test_encoded)

if len(rf_shap_vals.shape) == 3:
  shap.plots.bar(rf_shap_vals[:, :, 1])
  shap.plots.beeswarm(rf_shap_vals[:, :, 1])
else:
  shap.plots.bar(rf_shap_vals)
  shap.plots.beeswarm(rf_shap_vals)


# ==========================================
# 5. NEURAL NETWORK (MLP) CLASSIFIER
# ==========================================
print("\n=== Neural Network (MLP) Model ===")
nn_mod = MLPClassifier(
    hidden_layer_sizes=(20, 20), max_iter=3000, random_state=2
)
nn_mod.fit(X_train_encoded, y_train)

# Evaluasi Model Neural Network (MLP)
y_pred_nn = nn_mod.predict(X_test_encoded)
print("\n=== Evaluasi Neural Network (MLP) Kategorik ===")
print(metrics.classification_report(y_test, y_pred_nn))

def nn_model_proba(x):
  return nn_mod.predict_proba(x)[:, 1]


explainer_nn = shap.Explainer(nn_model_proba, Xbackground_encoded, seed=3454)
nn_shap_vals = explainer_nn(X_test_encoded)

shap.plots.bar(nn_shap_vals)
shap.plots.beeswarm(nn_shap_vals)